In [2]:
# 读取环境变量

import dotenv

dotenv.load_dotenv()
import os

DEEPSEEK_API = os.getenv("DEEPSEEK_API")

In [3]:
# 创建子智能体

import operator
from typing import Annotated, List, TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

model = init_chat_model(
    model="openai:deepseek-v4-pro", base_url="https://api.deepseek.com/v1", api_key=DEEPSEEK_API, temperature=0.7
)


class TextEditorState(TypedDict):
    messages: Annotated[List[AnyMessage], operator.add]


# 一个节点只返回一条 Message ？
def rewtite_node(state: TextEditorState) -> TextEditorState:
    reponse = model.invoke([SystemMessage(content="你是一名中文文本润色助手, 能将文字文言文化...")] + state["messages"])
    return {"messages": [reponse]}


graph_builder = StateGraph(TextEditorState)

graph_builder.add_node("rewtite_node", rewtite_node)

graph_builder.add_edge(START, "rewtite_node")
graph_builder.add_edge("rewtite_node", END)

text_editor_agent = graph_builder.compile()

# 测试
text_editor_agent.invoke({"messages": [HumanMessage(content="请将以下文字进行润色：今天天气真不错， 适合出去玩。")]})

{'messages': [HumanMessage(content='请将以下文字进行润色：今天天气真不错， 适合出去玩。', additional_kwargs={}, response_metadata={}),
  AIMessage(content='今日天色殊佳，正宜出游。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 372, 'prompt_tokens': 35, 'total_tokens': 407, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 362, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-pro', 'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402', 'id': 'ffbf8420-3bd7-488e-ab0b-9b9558a4b8c3', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f7b1d-27d1-7431-88be-09d4941eac26-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 372, 'total_tokens': 407, 'input_token_details': {'cache_read': 0}, 

In [ ]:
# 外部Agent

from deepagents import CompiledSubAgent, create_deep_agent
from langchain.chat_models import init_chat_model

text_helper_subagent = CompiledSubAgent(
    name="text-helper",
    description="""
    1. 用于中文文本润色
    2. 使用文言文
    """,
    runnable=text_editor_agent,
)

model_flash = init_chat_model(
    model="openai:deepseek-v4-flash", base_url="https://api.deepseek.com/v1", api_key=DEEPSEEK_API, temperature=0.7
)


agent = create_deep_agent(
    model=model_flash,
    system_prompt="""
    你是一个智能文本助手的协调者， 负责判断是否需要调用子智能体
    
    你的决定逻辑如下：
    1. 如何用户仅仅提出一般性问题，可以直接回答
    2. 如果用户提供了一段文本， 并且提出 "润色/优化/重写/精简"等需求时：
     - 请使用task工具， 将任务委派给 text-helper 子智能体处理
     - text-helper 子智能体返回处理后的文本
     - 你需要将润色后的文本原样返回给用户
    """,
    subagents=[text_helper_subagent],
)


result = agent.invoke({
    "messages": [
        HumanMessage(
            content="""请帮我润色如下文字：
            风和日丽，鸟语花香，在暮春的时光看小草的新绿，在温煦的春风中沐浴。也有野花芬芳馥郁，清新脱俗，待到山花烂漫时，她在丛中笑。更或者，羞涩的花瓣含苞待放，欲放含羞，微风中随绿色的荷叶轻轻摇曳婀娜的身姿，小荷才露尖尖角……
            """
        )
    ]
})

In [5]:
result["messages"][-1].content


'惠风和畅，鸟语花香。暮春时节，闲看芳草初碧，新绿茸茸；沐浴和煦春风，暖意融融。野花发于幽处，芬芳馥郁，清新绝俗，待得山花烂漫之时，伊人隐于丛中，嫣然浅笑。更有那含苞待放之花，羞涩欲语，娇红半敛，微风过处，随碧叶轻摇，婀娜生姿——正是小荷才露尖尖角，清香初透，无限幽情。'

In [7]:
result = agent.invoke({"messages": [HumanMessage(content="告诉我，你有哪些工具?")]})
from rich.console import Console
from rich.markdown import Markdown

Console().print(Markdown(result["messages"][-1].content))


我拥有的工具如下：                                                                                                 

 1 write_todos — 管理和跟踪复杂任务的待办事项列表，适合多步骤目标规划。                                            
 2 ls — 列出目录中的文件，用于探索文件系统。                                                                       
 3 read_file — 读取文件内容，支持分页（offset/limit），也支持读取图片、音视频、PDF等多模态文件。                   
 4 write_file — 创建并写入新文件。                                                                                 
 5 edit_file — 通过精确字符串替换编辑已有文件。                                                                    
 6 glob — 根据通配符模式（如 **/*.py）查找文件路径。                                                               
 7 grep — 在文件内容中搜索文本模式（支持文件名过滤和多种输出模式）。                                               
 8 task — 启动子智能体处理独立任务，支持以下子智能体类型：                                                         
    • general-purpose：通用型，可处理复杂研究、搜索、多步骤任务                                                    
    • text-helper：专门用于中文文本润色和文言文写作                                                                

有什么我可以帮你处理的吗？